<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week6_ExerciseXP_Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Initialisation des tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

text = "Hello, how are you today?"

# Utilisation de l'appel direct (recommandé) au lieu de encode_plus
# Tokenizing with BERT
bert_encoded = bert_tokenizer(text, add_special_tokens=True, return_tensors='pt')
print(f"BERT input_ids: {bert_encoded['input_ids']}")
print(f"BERT decoded: {bert_tokenizer.decode(bert_encoded['input_ids'][0])}")

# Tokenizing with XLM-RoBERTa
xlm_encoded = xlm_tokenizer(text, add_special_tokens=True, return_tensors='pt')
print(f"\nXLM-RoBERTa input_ids: {xlm_encoded['input_ids']}")
print(f"XLM-RoBERTa decoded: {xlm_tokenizer.decode(xlm_encoded['input_ids'][0])}")

BERT input_ids: tensor([[ 101, 7592, 1010, 2129, 2024, 2017, 2651, 1029,  102]])
BERT decoded: [CLS] hello, how are you today? [SEP]

XLM-RoBERTa input_ids: tensor([[    0, 35378,     4,  3642,   621,   398, 18925,    32,     2]])
XLM-RoBERTa decoded: <s> Hello, how are you today?</s>


In [ ]:
import pandas as pd
from transformers import BertTokenizer, XLMRobertaTokenizer
from sklearn.model_selection import StratifiedKFold

# Explicitly instantiating the tokenizer objects
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

text = "Hello, how are you today?"
max_len = 32

# Task 3: Preparing Input Data
# Using __call__ or encode_plus on the instance
encoded_data = bert_tokenizer(
    text,
    add_special_tokens=True,
    max_length=max_len,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

print(f"BERT Tokenizer loaded: {type(bert_tokenizer)}")
print(f"BERT Special Tokens: {bert_tokenizer.special_tokens_map}")

# Task 4 & 5: Loading Dataset and Creating Cross-Validation Folds
try:
    # Creating a dummy dataset for demonstration
    data = {
        'text': ["I love machine learning", "Transformers are great", "BERT is powerful", "AI is the future", "NLP is fun", "Data science is cool"],
        'label': [1, 1, 1, 0, 0, 0]
    }
    df = pd.DataFrame(data)
    print(f"\nDataset Shape: {df.shape}")

    # Implementing StratifiedKFold
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    folds = []

    for train_index, val_index in skf.split(df['text'], df['label']):
        train_df = df.iloc[train_index]
        val_df = df.iloc[val_index]
        folds.append((train_df, val_df))

    print(f"Successfully created {len(folds)} cross-validation folds.")
    print("First fold training sample label distribution:")
    print(folds[0][1]['label'].value_counts())

except Exception as e:
    print(f"Error: {e}")

BERT Tokenizer loaded: <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>
BERT Special Tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}

Dataset Shape: (6, 2)
Successfully created 3 cross-validation folds.
First fold training sample label distribution:
label
1    1
0    1
Name: count, dtype: int64


### Task 6: Custom PyTorch Dataset
We need a way to feed our tokenized data into the model. We'll create a class that inherits from `torch.utils.data.Dataset`.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Quick test with the first fold
train_df_fold0, val_df_fold0 = folds[0]

test_dataset = TextDataset(
    texts=train_df_fold0.text.to_numpy(),
    labels=train_df_fold0.label.to_numpy(),
    tokenizer=bert_tokenizer,
    max_len=max_len
)

test_loader = DataLoader(test_dataset, batch_size=2)

# Preview one batch
batch = next(iter(test_loader))
print(f"Keys in batch: {batch.keys()}")
print(f"Input IDs shape: {batch['input_ids'].shape}")
print(f"Labels: {batch['labels']}")

Keys in batch: dict_keys(['text', 'input_ids', 'attention_mask', 'labels'])
Input IDs shape: torch.Size([2, 32])
Labels: tensor([1, 1])


### Task 7: Initialize Model
We will load the pre-trained BERT model with a sequence classification head. We'll specify the number of labels (in this case, 2: binary classification).

In [ ]:
from transformers import BertForSequenceClassification
import torch.optim as optim

# Loading the model
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded on {device}")

# Setting up the optimizer using torch.optim.AdamW
optimizer = optim.AdamW(model.parameters(), lr=2e-5, eps=1e-8)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


### Task 8: Training Loop
We will define a training function that iterates through the DataLoader, performs forward and backward passes, and updates the model weights.

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(data_loader)

# Let's run a very quick training on the first fold to test
train_loader = DataLoader(test_dataset, batch_size=2)

print("Starting small test training...")
for epoch in range(3):
    avg_loss = train_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch + 1} | Loss: {avg_loss:.4f}")

Starting small test training...
Epoch 1 | Loss: 0.7850
Epoch 2 | Loss: 0.7173
Epoch 3 | Loss: 0.6128


### Task 9: Evaluation Function
This function will put the model in evaluation mode and compute the accuracy on the validation dataset.

In [ ]:
import numpy as np

def eval_model(model, data_loader, device):
    model.eval()
    correct_predictions = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            _, preds = torch.max(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

    return correct_predictions.double() / len(data_loader.dataset)

# Quick validation test
val_dataset = TextDataset(
    texts=val_df_fold0.text.to_numpy(),
    labels=val_df_fold0.label.to_numpy(),
    tokenizer=bert_tokenizer,
    max_len=max_len
)
val_loader = DataLoader(val_dataset, batch_size=2)

accuracy = eval_model(model, val_loader, device)
print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.0000


### Task 10: Complete Cross-Validation Loop
We will now iterate through all the folds we created, training and validating a fresh model instance for each fold.

In [ ]:
epochs = 3
fold_accuracies = []

for i, (train_df_fold, val_df_fold) in enumerate(folds):
    print(f"\n--- Training Fold {i+1} ---")

    # 1. Prepare DataLoaders
    train_ds = TextDataset(train_df_fold.text.to_numpy(), train_df_fold.label.to_numpy(), bert_tokenizer, max_len)
    val_ds = TextDataset(val_df_fold.text.to_numpy(), val_df_fold.label.to_numpy(), bert_tokenizer, max_len)

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2)

    # 2. Re-initialize model and optimizer for a fresh start on each fold
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=2e-5)

    # 3. Simple training loop for the fold
    for epoch in range(epochs):
        loss = train_epoch(model, train_loader, optimizer, device)

    # 4. Evaluation
    acc = eval_model(model, val_loader, device)
    fold_accuracies.append(acc.item())
    print(f"Fold {i+1} Accuracy: {acc:.4f}")

print(f"\nAverage CV Accuracy: {np.mean(fold_accuracies):.4f}")


--- Training Fold 1 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 1 Accuracy: 0.5000

--- Training Fold 2 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 2 Accuracy: 0.5000

--- Training Fold 3 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 3 Accuracy: 0.5000

Average CV Accuracy: 0.5000
